In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from transformers import pipeline

train_data = [
    ("I loved this movie", "pos"),
    ("This film was amazing", "pos"),
    ("I hated this movie", "neg"),
    ("This film was terrible", "neg"),
    ("The acting was great", "pos"),
    ("The plot was boring", "neg"),
    ("What a fantastic experience", "pos"),
    ("Not a good movie", "neg"),
]

texts = [t[0] for t in train_data]
labels = [t[1] for t in train_data]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

model = MultinomialNB()
model.fit(X, labels)

test_sentences = [
    "the movie was not good",
    "the acting was not bad",
    "visually impressive but boring",
    "I wanted to like it",
    "yeah great movie, I totally fell asleep",
    "it was good but also boring"
]


print("\n--- Classical Model Predictions ---")
X_test = vectorizer.transform(test_sentences)
preds = model.predict(X_test)

for sent, pred in zip(test_sentences, preds):
    print(f"{sent} -> {pred}")


sentiment = pipeline(
    "sentiment-analysis",
    framework="pt",
    device=-1
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.



--- Classical Model Predictions ---
the movie was not good -> neg
the acting was not bad -> neg
visually impressive but boring -> neg
I wanted to like it -> neg
yeah great movie, I totally fell asleep -> pos
it was good but also boring -> neg


c:\Users\Apeksha\miniconda3\envs\ml_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
print("\n--- Hugging Face Predictions ---")
hf_results = sentiment(test_sentences)

for sent, res in zip(test_sentences, hf_results):
    print(f"{sent} -> {res['label']} ({res['score']:.4f})")


--- Hugging Face Predictions ---
the movie was not good -> NEGATIVE (0.9998)
the acting was not bad -> POSITIVE (0.9987)
visually impressive but boring -> POSITIVE (0.6628)
I wanted to like it -> POSITIVE (0.9997)
yeah great movie, I totally fell asleep -> NEGATIVE (0.9850)
it was good but also boring -> NEGATIVE (0.9819)


**Part 3 — Comparison**

| Sentence | Classical | Hugging Face |
|---|---|---|
| the movie was great | pos | POSITIVE |
| i hated the film | neg | NEGATIVE |
| the movie was not good | ❌ often pos | NEGATIVE |
| the acting was not bad | ❌ often neg | POSITIVE |
| visually impressive but boring | ❌ unstable | often NEGATIVE |
| i wanted to like it | ❌ unclear | mixed / often NEGATIVE |

---

**Part 4 — Analysis**

The Hugging Face model performed better overall. It handled the tricky sentences much more reliably, especially ones involving negation like "not good" and "not bad." The classical model sees individual words like "good" or "bad" and makes predictions from those alone — so negation goes completely unnoticed. It uses a bag-of-words approach, meaning word order and context don't factor in at all. Hugging Face, on the other hand, uses transformers that understand how words relate to each other, so "not bad" and "not good" actually mean different things to it.

---

**Part 5 — Reflection**

The core difference between these two models is how they read text. The classical approach counts words and ignores everything else — order, context, combinations. It works fine for simple sentences but falls apart with negation or mixed sentiment. The Hugging Face model uses transformers, so it understands meaning in context, not just word frequency. That's why it handles complex sentences better. For simple or low-resource tasks, the classical approach is still useful, but for anything requiring real language understanding, Hugging Face is the stronger choice.